# Demo Text-to-SQL con Snowflake Cortex Analyst

**Master AI – Clases IA**

> ℹ️ **Esto es una guía, no un notebook para “ejecutar”.** Casi todos los pasos
> (cargar los CSV, crear la Semantic View, desplegar la app) se hacen a mano en
> **Snowsight**, la web de Snowflake. Las celdas de código que verás son SQL /
> Python de **referencia** para pegar en Snowsight, no para correr aquí en
> Jupyter. La única celda que sí puedes ejecutar en local es la del paso 1, que
> solo inspecciona los CSV con pandas.

> 🆓 **Necesitas una cuenta de Snowflake (gratis):** entra en
> [signup.snowflake.com](https://signup.snowflake.com/) y crea una cuenta de
> prueba. El **trial gratuito** dura **30 días** e incluye **$400 de créditos**
> para gastar, **sin necesidad de meter tarjeta de crédito**. Elige la edición
> *Enterprise* y un proveedor cloud (AWS, Azure o GCP) en una región cercana.
> Con eso tienes de sobra para todo este lab (Cortex Analyst + Streamlit in
> Snowflake).

En este notebook construirás, paso a paso, una demo de **text-to-SQL**: desde
los ficheros CSV originales hasta una app de Streamlit que responde preguntas
en lenguaje natural usando Cortex Analyst.

La idea: cargas unos datos de ventas en Snowflake, los describes con una
**Semantic View** y dejas que Cortex Analyst traduzca preguntas como
*"compara ingresos reales vs previstos por mes"* en SQL, ejecute la consulta y
muestre el resultado.

Trabajarás sobre **Snowsight** (la interfaz web de Snowflake) para los pasos de
carga y de creación de la Semantic View, y sobre **Snowflake Notebooks /
Streamlit in Snowflake** para el resto.

> 📚 **Guía oficial de Snowflake (por si quieres seguir el tutorial original):**
> [Tutorial: Answer questions about time-series revenue data with Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/tutorials/tutorial-1)
> · Documentación de [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

## Índice
1. Punto de partida: los datos
2. Crear las tablas arrastrando los CSV (sin SQL)
3. Revisar los datos cargados
4. Crear la Semantic View para Cortex Analyst
5. Probar Cortex Analyst desde SQL
6. La app de Streamlit (text-to-SQL end-to-end)
7. Resumen del flujo completo


## 1. Punto de partida: los datos

Partimos de tres ficheros CSV de un dataset de **ingresos diarios** (serie
temporal de ventas):

| Fichero | Tipo | Tabla destino |
|---|---|---|
| `daily_revenue.csv` | Tabla de hechos (*fact*) | `daily_revenue` |
| `product.csv` | Dimensión | `product_dim` |
| `region.csv` | Dimensión | `region_dim` |

**Columnas de cada CSV:**

- `daily_revenue.csv` → `DATE, REVENUE, COGS, FORECASTED_REVENUE, Product_id, Region_id`
- `product.csv` → `Product_id, Product_line`
- `region.csv` → `Region_id, Region, State`

El modelo es un **esquema en estrella**: una tabla de hechos (`daily_revenue`)
que se une a dos dimensiones (`product_dim`, `region_dim`) mediante `Product_id`
y `Region_id`.

> Todos los CSV usan **coma** `,` como separador y traen cabecera en la
> primera fila.


In [ ]:
# Inspecciona los CSV antes de subirlos a Snowflake.
import pandas as pd

daily = pd.read_csv("cortex_data/daily_revenue.csv")
product = pd.read_csv("cortex_data/product.csv")
region = pd.read_csv("cortex_data/region.csv")

print("daily_revenue:", daily.shape)
print(daily.head(3), "\n")
print("product:", product.shape)
print(product, "\n")
print("region:", region.shape)
print(region.head(5))

## 2. Crear las tablas arrastrando los CSV (sin SQL)

No hace falta escribir nada de SQL para cargar los datos. En **Snowsight**,
Snowflake crea la tabla por ti, detecta las columnas y carga el CSV, todo
arrastrando el fichero.

Lo haremos **uno a uno** con los tres CSV:

| Fichero | Nombre de tabla que pondrás |
|---|---|
| `daily_revenue.csv` | `DAILY_REVENUE` |
| `product.csv` | `PRODUCT_DIM` |
| `region.csv` | `REGION_DIM` |


### Pasos en Snowsight (repite para cada CSV)

1. En el menú lateral, entra en **Data » Databases** y abre tu base de datos y
   tu esquema (por ejemplo `DEMO.PUBLIC`).
2. Pulsa el botón **Create » Table » From File**
   *(o, dentro del esquema: pestaña Tables → **+ Table** → **From File**).*
3. **Arrastra el CSV** (o pulsa *Browse* y selecciónalo).
4. Escribe el **nombre de la tabla** según la tabla de arriba
   (`DAILY_REVENUE`, `PRODUCT_DIM` o `REGION_DIM`).
5. Snowflake muestra una **vista previa** y detecta automáticamente las
   columnas y sus tipos. Revísala rápidamente.
6. Pulsa **Next** y luego **Load**.

Repite los pasos **1–6** con los otros dos CSV. Al terminar tendrás las tres
tablas creadas y con datos, sin haber escrito SQL.

> 💡 Snowflake infiere los tipos a partir del CSV. Si alguna columna numérica
> sale como texto, puedes ajustar el tipo en la misma pantalla de vista previa
> antes de pulsar *Load*.


## 3. Revisar los datos cargados (opcional)

Para comprobar que las tablas tienen datos puedes abrir cada tabla en
**Data » Databases** y mirar la pestaña **Data Preview** — tampoco necesitas
SQL aquí.

Si prefieres comprobarlo con una consulta rápida, puedes usar (cambia
`DEMO.PUBLIC` por tu base de datos y esquema):


In [ ]:
SELECT COUNT(*) AS filas_daily FROM DEMO.PUBLIC.DAILY_REVENUE;
SELECT * FROM DEMO.PUBLIC.DAILY_REVENUE LIMIT 5;

SELECT * FROM DEMO.PUBLIC.PRODUCT_DIM;
SELECT * FROM DEMO.PUBLIC.REGION_DIM LIMIT 5;


## 4. Crear la Semantic View para Cortex Analyst

La **Semantic View** (o *semantic model*) es la capa que permite a Cortex
Analyst entender el significado de las tablas: qué es una dimensión, qué es una
métrica, cómo se relacionan las tablas y qué sinónimos en lenguaje natural usar.

Créala desde Snowsight: → **AI & ML** → **Cortex Analyst** → crear una nueva
Semantic View. Define:

- **Tablas lógicas:** `daily_revenue` (hechos), `product_dim`, `region_dim`.
- **Relaciones (joins):** `daily_revenue.product_id = product_dim.product_id`
  y `daily_revenue.region_id = region_dim.region_id`.
- **Dimensiones:** `date`, `product_line`, `sales_region`, `state`.
- **Métricas / facts:** `revenue`, `cogs`, `forecasted_revenue`.
- **Sinónimos:** p. ej. *"ingresos"/"facturación"* → `revenue`,
  *"previsión"* → `forecasted_revenue`.

> 💡 La calidad de las respuestas depende de esta capa: cuanto mejor definas
> dimensiones, métricas, joins y sinónimos, mejor SQL generará Cortex.

### Nombre de la Semantic View

El nombre completo es `<BASE_DE_DATOS>.<ESQUEMA>.<NOMBRE_VISTA>` y depende de
qué base de datos y esquema elijas al crearla. **Apunta el nombre que le
pongas**, porque tendrás que escribirlo en la variable `SEMANTIC_VIEW` de
`app.py` (paso 6). Si ambos no coinciden, la app no encontrará la Semantic View.

Para mantenerlo ordenado, puedes crear la Semantic View en la misma base de
datos y esquema que las tablas (`cortex_analyst_demo.revenue_timeseries`).


## 5. Probar Cortex Analyst desde SQL

Cortex Analyst se consume vía **API REST** (`/api/v2/cortex/analyst/message`).
La respuesta incluye el **SQL generado**, que luego se ejecuta normalmente.
El ciclo es:

1. Pregunta en lenguaje natural →
2. Cortex Analyst (con la Semantic View) genera SQL →
3. Snowflake ejecuta el SQL →
4. Se muestran tabla + gráfico.

Ejemplo del tipo de SQL que Cortex genera por debajo para
*"compara ingresos reales vs previstos por mes"*:


In [ ]:
SELECT
    DATE_TRUNC('month', date) AS mes,
    SUM(revenue)              AS ingresos_reales,
    SUM(forecasted_revenue)  AS ingresos_previstos
FROM cortex_analyst_demo.revenue_timeseries.daily_revenue
GROUP BY 1
ORDER BY 1;

## 6. La app de Streamlit (text-to-SQL end-to-end)

La pieza final es una app de **Streamlit in Snowflake** (`app.py`). El usuario
escribe una pregunta, la app llama a Cortex Analyst por REST, ejecuta el SQL
devuelto y pinta el resultado. Al ejecutarse dentro de Snowflake usa
`get_active_session()`, así que no hace falta poner usuario ni contraseña en el
código.

Qué hace cada función clave:

- `get_active_session()` → usa la sesión de Snowflake ya autenticada.
- `get_session_token()` → obtiene el token para llamar a la API REST.
- `ask_cortex(question)` → envía la pregunta + la Semantic View a Cortex Analyst.
- `run_sql(sql)` → ejecuta el SQL devuelto y lo trae como DataFrame.
- `show_cortex_response(...)` → muestra texto, SQL, tabla y gráfico.

Extracto de la lógica central (el código completo está en `app.py`):


```python
import pandas as pd
import requests
import streamlit as st
from snowflake.snowpark.context import get_active_session

SNOWFLAKE_HOST = "hb18782.eu-west-3.aws.snowflakecomputing.com"
SEMANTIC_VIEW = "DEMO.PUBLIC.DEMO_EBIS"
CORTEX_ANALYST_ENDPOINT = "/api/v2/cortex/analyst/message"

session = get_active_session()


def get_session_token() -> str:
    conn = session.connection
    rest = getattr(conn, "rest", None)
    if rest is not None and hasattr(rest, "token"):
        return rest.token
    inner_conn = getattr(conn, "_conn", None)
    if inner_conn is not None:
        inner_rest = getattr(inner_conn, "rest", None)
        if inner_rest is not None and hasattr(inner_rest, "token"):
            return inner_rest.token
    raise RuntimeError("No se pudo obtener el token de sesión de Snowflake.")


def ask_cortex(question: str) -> dict:
    token = get_session_token()
    url = f"https://{SNOWFLAKE_HOST}{CORTEX_ANALYST_ENDPOINT}"
    request_body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": question}]}
        ],
        "semantic_view": SEMANTIC_VIEW,
    }
    headers = {
        "Authorization": f'Snowflake Token="{token}"',
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, json=request_body, timeout=60)
    if response.status_code >= 400:
        raise RuntimeError(response.text)
    result = response.json()
    result["request_id"] = response.headers.get("X-Snowflake-Request-Id")
    return result


def run_sql(sql: str) -> pd.DataFrame:
    return session.sql(sql).to_pandas()
```

### ⚠️ Qué tienes que cambiar tú en `app.py`

Estas dos líneas son **distintas para cada uno** y hay que personalizarlas, o la
app no funcionará:

```python
SNOWFLAKE_HOST = "hb18782.eu-west-3.aws.snowflakecomputing.com"  # ← TU host
SEMANTIC_VIEW  = "DEMO.PUBLIC.DEMO_EBIS"                          # ← TU Semantic View
```

**1) `SNOWFLAKE_HOST` — el host de TU cuenta de Snowflake**

Es la dirección de tu cuenta y cambia según la organización y la región. Cómo
obtenerlo:

- En Snowsight, abajo a la izquierda, pulsa sobre tu **cuenta/usuario** →
  **Account** → copia el **Account/Server URL**, o
- ejecuta en un worksheet:
  ```sql
  SELECT CURRENT_ACCOUNT() AS cuenta,
         CURRENT_REGION()  AS region;
  ```
- El host tiene la forma `<identificador_cuenta>.snowflakecomputing.com`
  (por ejemplo `abc12345.eu-west-1.aws.snowflakecomputing.com`).
  Pégalo **sin** `https://` y **sin** barra final.

**2) `SEMANTIC_VIEW` — el nombre completo de TU Semantic View**

Es `BASE_DE_DATOS.ESQUEMA.NOMBRE_VISTA` y depende de dónde y con qué nombre la
creaste en el **paso 4**. Por ejemplo, si la creaste en `DEMO.PUBLIC` con el
nombre `DEMO_EBIS`, sería `DEMO.PUBLIC.DEMO_EBIS`.

> 💡 Si la app da error de "semantic view not found", casi siempre es porque
> este nombre no coincide exactamente (mayúsculas/minúsculas, BD o esquema
> distintos) con la Semantic View real.

El resto del `app.py` (token, llamada REST, ejecución del SQL, visualización e
interfaz) **no hace falta tocarlo**.


### Cómo desplegar la app en Snowflake

1. Snowsight → **Projects** → **Streamlit** → **+ Streamlit App**.
2. Elige base de datos/esquema y un warehouse (`cortex_analyst_wh`).
3. Pega el contenido de `app.py`.
4. Pulsa **Run** y prueba con las preguntas de ejemplo de la barra lateral.


## 7. Resumen del flujo completo

```
3 CSV (daily_revenue, product, region)
        │  (Snowsight » Create Table » From File — arrastrar cada CSV)
        ▼
Tablas creadas automáticamente
   (DAILY_REVENUE + PRODUCT_DIM + REGION_DIM)
        │  (Semantic View vía Cortex Analyst UI)
        ▼
Semantic View  (capa semántica: dimensiones, métricas, joins, sinónimos)
        │
        ▼
App Streamlit ──► Cortex Analyst (REST) ──► genera SQL
        ▲                                        │
        │                                        ▼
   tabla + gráfico ◄──── Snowflake ejecuta el SQL
```

**En una frase:** cargamos 3 CSV a Snowflake arrastrándolos (Create Table → From File), los describimos
con una Semantic View, y Cortex Analyst traduce preguntas en lenguaje natural
a SQL que Streamlit muestra como tabla y gráfico.
